<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #FFFFFF; max-width: 90%; overflow-x: auto;">

<img src="../../resources/swdb_logo.jpg">

<h1 align="center">Mini Workshop 4: Co-modulation or coincidental drift?</h1>
<h3 align="center">Summer Workshop on the Dynamic Brain</h3>
<h4 align="center">Thursday, August 27th, 2026</h4>
<h4 align="center">Day 4</h4>

---

***Authors:** Nick Steinmetz, Carrie Stine*

---

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto;">

## The Setup:

Imagine an experiment where we will record from a population of 100 slowly fluctuating neurons during a task that alternates between two conditions or **blocks**, which we will call +1 or -1. Each block consists of roughly 30-70 trials, each trial lasts for 1 second, and the full session lasts about 500 trials. Alongside the neural recording, we are also tracking the animal's **pupil diameter** as a measure of arousal.

In [ ]:
# Setup (imports and plotting defaults)
import os
os.chdir('/code/mini-workshops/nb4')
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from rastermap import Rastermap
import logging

logging.getLogger('rastermap').setLevel(logging.WARNING)

# Global figure settings
plt.rcParams['font.family']       = 'sans-serif'
plt.rcParams['font.sans-serif']   = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['pdf.fonttype']      = 42
plt.rcParams['ps.fonttype']       = 42


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto;">

## Step 1

Load the data and look at the contents. NOTE: This dataset is identical to the simulated data from Workshop 2!

In [ ]:
# Load the dataset
d            = np.load('data/simulation.npz')
fr           = d['fr']
spikes       = d['spikes']
block_values = d['block_values']
block_ids    = d['block_ids']
pupil        = d['pupil']
t_pts        = d['t_pts']
t_trial      = d['t_trial']
DT           = float(d['DT'])
TRIAL_DUR    = float(d['TRIAL_DUR'])
per_trial    = int(d['per_trial'])
n_neurons, n_trials = fr.shape
n_blocks = len(np.unique(block_ids))

frz = (fr - fr.mean(axis=1, keepdims=True)) / fr.std(axis=1, keepdims=True)

In [ ]:
print(f'{n_neurons} neurons, {n_trials} trials ({n_trials * TRIAL_DUR / 60:.1f} min), {n_blocks} blocks')
print(f'Firing rates: {fr.min():.1f} - {fr.max():.1f} spikes/s (mean {fr.mean():.1f})')
print(f'{spikes.sum():,} spikes total')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto;">

## Step 2

Generate a firing rate matrix (neurons x trials) and z-score it.

In [ ]:
# sum spikes within each trial for each neuron
counts = spikes.reshape(n_neurons, n_trials, per_trial).sum(axis=2)   # (neurons, trials)
fr     = counts / TRIAL_DUR                                            # convert to spikes/s
frz    = (fr - fr.mean(axis=1, keepdims=True)) / fr.std(axis=1, keepdims=True)  # z-score

def show_matrix(ax, M, cmap, vmin, vmax, title, cbar_label, ylabel='neuron'):
    """Display a neurons x trials matrix. interpolation='nearest' prevents matplotlib
    from blurring across trials or neurons, which would invent structure."""
    im = ax.imshow(M, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation='nearest', extent=[0, M.shape[1], M.shape[0], 0])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label=cbar_label, fraction=0.025, pad=0.01)
    return im

fig, axes = plt.subplots(2, 1, figsize=(9.5, 6), sharex=True)
show_matrix(axes[0], fr, 'magma', 0, np.percentile(fr, 99.5),
            'Spike counts binned by trial', 'firing rate (spikes/s)')
show_matrix(axes[1], frz, 'RdBu_r', -3, 3,
            'Same matrix, z-scored per neuron', 'z-score')
axes[1].set_xlabel('trial')
plt.tight_layout()
plt.show()


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF;  max-width: 90%; overflow-x: auto;">

## Step 3 - Sort activity with Rastermap
[Rastermap](https://github.com/MouseLand/rastermap) sorts the rows such that
neurons with similar activity patterns end up adjacent. On data *without* slow
fluctuations this sorts neurons by their tuning properties. On our simulated
data, it sorts them by which phase of the slow fluctuation each neuron happens
to be in.

</div>


In [ ]:
def rastermap_order(M):
    """Neuron ordering from Rastermap fit on matrix M (neurons x trials).
    Returns isort: an integer array of neuron indices in sorted order."""
    return Rastermap(n_clusters=10, n_PCs=32, locality=0.5, time_lag_window=0, verbose=False).fit(M).isort


# use rastermap to sort the z-scored spiking data (frz) by similarity
isort = rastermap_order(frz)

# plot the sorted matrix of neurons x trials
fig, ax = plt.subplots(figsize=(9.5, 3.6))
show_matrix(ax, frz[isort], 'RdBu_r', -3, 3,
            'Rastermap-sorted activity (sort and display use the same data)',
            'z-score', ylabel='neuron (Rastermap order)')
ax.set_xlabel('trial')
plt.tight_layout()
plt.show()


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF;  max-width: 90%; overflow-x: auto;">

## Step 4 - Cross-validate the sort

To test whether the sorted structure is real, we should fit the sort on *some*
trials and evaluate it on *different* trials.


</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF;  max-width: 90%; overflow-x: auto;">

#### Helper functions:
- `neighbour_corr` measures how well the sort captured shared structure: it
computes the mean Pearson correlation between adjacent neurons in the sorted order,
evaluated on a matrix `M`.

- `mean_pair_corr` measures the mean correlation across all possible neuron pairs. 
It represents the baseline you would get if you picked two neurons at random, regardless 
of where they ended up in the sort.

- `plot_split` turns our matrix plotting code for into a reusable function.

</div>

In [ ]:
def neighbour_corr(order, M):
    """Mean Pearson r between neurons that `order` placed adjacent to each other,
    measured on matrix M. High on fitting data, stays high if structure is real."""
    C = np.corrcoef(M)
    return np.mean(C[order[:-1], order[1:]])

def mean_pair_corr(M):
    """Mean Pearson r over all neuron pairs (baseline for neighbour_corr)."""
    C = np.corrcoef(M)
    return np.mean(C[np.triu_indices(len(C), 1)])

def plot_split(order, fit_idx, held_idx, fit_label, held_label):
    """Plot a neurons x trials matrix for both the fitting and held-out trial sets,
    using the neuron ordering `order` derived from `fit_idx` trials only."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    show_matrix(axes[0], frz[order][:, fit_idx],  'RdBu_r', -3, 3,
                f'Sorted on {fit_label}, showing {fit_label}',
                'z-score', ylabel=f'neuron ({fit_label} order)')
    show_matrix(axes[1], frz[order][:, held_idx], 'RdBu_r', -3, 3,
                f'Same sort order, showing held-out {held_label}',
                'z-score', ylabel=f'neuron ({fit_label} order)')
    axes[0].set_xlabel(fit_label)
    axes[1].set_xlabel(held_label)
    plt.tight_layout()
    plt.show()
    

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF;  max-width: 90%; overflow-x: auto;">

### Interleaved split (odd vs even trials)

 Train the row sorting on every other trial (odd trials), then evaluate the resulting structure on the remaining trials (even trials). 

 </div>

In [ ]:
# 1. Split the trials into interleaved sets (odd vs even):
odd, even = np.arange(1, n_trials, 2), np.arange(0, n_trials, 2)

# 2. Fit the sort using only the odd trials:
order_odd = rastermap_order(frz[:, odd])

# 3. Plot the sorted fit on both the odd and even trials independently:
plot_split(order_odd, odd, even, 'odd trials', 'even trials')

# 4. Compute the neighbour_corr on both sets of trials and the mean_pair_corr on the held-out even trials:
neighbour_odd = neighbour_corr(order_odd, frz[:, odd])
neighbour_even = neighbour_corr(order_odd, frz[:, even])
meanpair_even = mean_pair_corr(frz[:, even])

print('Interleaved Split:')
print(f'Fit (odd trials):       neighbour_corr = {neighbour_odd:+.3f}')
print(f'Held-out (even trials): neighbour_corr = {neighbour_even:+.3f}')
print(f'Baseline (all pairs):   mean_pair_corr = {meanpair_even:+.3f}')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF;  max-width: 90%; overflow-x: auto;">

The rastermap shows clear structure, with sets of neurons having similar blocks of high or low activity across trials. Cross-validation confirmed the effect; if we train the sort on half of the trials, we see similar structure and high correlation between neighbors on the held-out half of trials. A colleague concludes:

> *"These neurons are clearly co-modulated."*

**This conclusion is not supported by the analysis.** Work
through the exercise below to find out why.


</div>

<div style="border-left: 3px solid #07bc0a; padding: 1px; padding-left: 10px; background: #DFF0D8; max-width: 90%; overflow-x: auto; color: #000000;">

### Exercise: Cross-validate the sorted activity

<ol>

<li><strong>Design a fair test.</strong>  What other ways could you split the test/train sets for cross-validation?
<details>
<summary>Hint</summary>

*A neuron's slow fluctuation during the first half of the session will be different in the second half. Train the fit on trials from the **first half** of the session, then evaluate the resulting structure on the trials from the **second half**.*
 1. *Split the trials into first half vs second half of the full session*
 2. *Fit the sort on trials from the first half of the session*
 3. *Plot the resulting fit on both the training and testing trials independently*
 4. *Compute `neighbour_corr` on the held-out trials from the second half of the session*

</details>
</li>

</ol>

</div>


In [ ]:
# YOUR CODE HERE


<div style="border-left: 3px solid #f0b429; padding: 1px; padding-left: 10px; background: #FFF9C4;  max-width: 90%; overflow-x: auto; color: #000000;">

## Key takeaways

<details>
<summary><b>reveal after completing the exercises!</b></summary>

The slow fluctuation of each neuron during the first half of trials is not the same as during the second half. Interleaving the trials in method 1 was **not** a test of the sort. Neighbouring trials share the same slow fluctuation, so the held-out half carries the same chance correlations the sort was fit to, and the structure replicates. Splitting the session into contiguous halves **is** a test, and the similarity ordering does not transfer.

</details>

<br>

</div>